<a href="https://colab.research.google.com/github/foolishcoder1/23f2004185_git_challenge_lab/blob/main/Milestone4_Solver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DLGenAI Project 26T3 - Milestone 4 solver
Run top to bottom on Kaggle (data attached, GPU on) or Colab. Every answer is printed **in the form's "Answer format"**.

| Cells | Questions | Needs GPU? |
|---|---|---|
| 1-2 | working subset, split | no |
| 3 | Q1, Q2, Q3 (imbalance, pos_weight, sqrt sampler) | no |
| 4 | Q4, Q5, Q6, Q7 part B (scheduler, early stopping, memory, naive Dice) | no |
| 5 | Q7 part A, Q9 (all-empty baseline, defect-size bins) | no |
| 6-7 | Q8, Q10 (train 4 short runs: R0, R0b, R1 = Q8 run, R2) | yes (~10 min on a T4) |

**Official score (confirmed in the milestone text):** `0.70*FgDice + 0.20*FPscore + 0.10*EmptyAcc`, `FPscore = 1 - mean(|P|/102400)` over empty pairs.
Set `USE_WANDB = True` in cell 1 to log every run to `23f2004185-t32026` (the milestone requires it).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os, glob, subprocess, sys, math, random, time, json, copy
from collections import Counter
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

USE_WANDB = True                    # True = log every training run to W&B (required by the milestone)
WANDB_PROJECT = '23f2004185-t32026'
H, W = 128, 800; CANVAS = H * W
DEFECTS = ['defect_a', 'defect_b', 'defect_c', 'defect_d']

# Detect if we are actually in a Kaggle Environment
IN_KAGGLE = os.path.exists('/kaggle/input') and not os.path.exists('/content')
root = os.environ.get('SG_DATA_ROOT') or ('/kaggle/input' if IN_KAGGLE else '/content/dataset')

def find_data_dir(r):
    for p in glob.glob(f'{r}/**/train.csv', recursive=True):
        if os.path.exists(os.path.join(os.path.dirname(p), 'sample_submission.csv')): return os.path.dirname(p)

if find_data_dir(root) is None and not IN_KAGGLE:      # Colab: download with the Kaggle API (accept the rules first; token in Colab Secrets)
    from google.colab import userdata
    for k in ('KAGGLE_USERNAME', 'KAGGLE_KEY', 'KAGGLE_API_TOKEN'):
        try: os.environ[k] = userdata.get(k)
        except Exception: pass
    os.makedirs(root, exist_ok=True)
    try:
        subprocess.run(['kaggle', 'competitions', 'download', '-c', 'industrial-surface-defect-detection', '-p', root], check=True)
        subprocess.run(f'cd {root} && unzip -q -o "*.zip"', shell=True, check=True)
    except Exception as e:
        print("Failed downloading dataset via Kaggle API. Please ensure your Kaggle credentials are set in Colab Secrets.", e)

# Fallback search in mounted Google Drive if local download failed or isn't used
DATA_DIR = find_data_dir(root) or find_data_dir('/content/drive/MyDrive/public')
assert DATA_DIR, 'competition data not found. Please check your Kaggle API key in Colab Secrets or ensure your Google Drive contains the dataset.'
print('data:', DATA_DIR)

Failed downloading dataset via Kaggle API. Please ensure your Kaggle credentials are set in Colab Secrets. Command '['kaggle', 'competitions', 'download', '-c', 'industrial-surface-defect-detection', '-p', '/content/dataset']' returned non-zero exit status 1.
data: /content/drive/MyDrive/public


## Common setup: 1,200-image working subset, decoded masks, 80/20 stratified split

In [6]:
df = pd.read_csv(f'{DATA_DIR}/train.csv')
ids = sorted(df.image_id.unique())[:1200]                      # unique ids, ASCII order, first 1,200
sub = df[df.image_id.isin(ids)]; assert len(sub) == 4800, f'expected 4800 rows, got {len(sub)}'
idx_of = {k: i for i, k in enumerate(ids)}; N = len(ids)

M = np.zeros((N, 4, H, W), np.uint8)                           # decoded masks, native 128x800 canvas
for r in sub.itertuples():
    if isinstance(r.mask_rle, str) and r.mask_rle.strip():
        s = np.array(r.mask_rle.split(), dtype=np.int64); flat = np.zeros(CANVAS, np.uint8)
        for a, b in zip(s[0::2] - 1, s[1::2]): flat[max(a, 0):min(a + b, CANVAS)] = 1     # clip runs that exceed the canvas
        M[idx_of[r.image_id], DEFECTS.index(r.defect_type)] = flat.reshape((H, W), order='F')   # Fortran order, 1-based starts
area = M.sum((2, 3)).astype(np.int64); pos = area > 0           # [N,4] defect area per pair, and positivity
combo = np.array(['+'.join(d for d, p in zip(DEFECTS, row) if p) or 'clean' for row in pos])
cnt = Counter(combo)
merged = np.array([c if cnt[c] >= 3 else 'rare_other' for c in combo])
print('rare_other images:', int((merged == 'rare_other').sum()), '(must be >= 2 for a stratified split)')
tr_idx, va_idx = train_test_split(np.arange(N), test_size=0.2, stratify=merged, random_state=42)
print('train / validation images:', len(tr_idx), '/', len(va_idx))

rare_other images: 7 (must be >= 2 for a stratified split)
train / validation images: 960 / 240


## Q1-Q3: imbalance at image, pixel and sampling level

In [8]:
# ---- Q1 ----
n_clean = cnt['clean']; b = pos.sum(0); sizes = sorted(cnt.values())
print('Q1:', f'{n_clean}; {b[0]}, {b[1]}, {b[2]}, {b[3]}; {len(cnt)}; {sizes[-1]}, {sizes[0]}')

# ---- Q2 (defect_b = column 1) ----
tot_b = int(M[:, 1].sum()); n_b = int(pos[:, 1].sum())
print('Q2:', f'{tot_b / (N * CANVAS) * 100:.4f}; {tot_b / n_b:.1f}; {round((N * CANVAS - tot_b) / tot_b)}')

# ---- Q3: bounded square-root sampler on the 1,200-image combination groups (no merging) ----
w = {g: min(math.sqrt(n_clean / n), 5.0) for g, n in cnt.items()}
tot_w = sum(n * w[g] for g, n in cnt.items()); has_b = lambda g: 'defect_b' in g.split('+')
a_s, a_n = n_clean * w['clean'] / tot_w * 100, n_clean / N * 100
b_s = 1200 * sum(n * w[g] for g, n in cnt.items() if has_b(g)) / tot_w
b_n = 1200 * sum(n for g, n in cnt.items() if has_b(g)) / N
capped = sum(1 for g, n in cnt.items() if math.sqrt(n_clean / n) >= 5)
print('Q3:', f'{a_s:.2f}, {a_n:.2f}; {b_s:.2f}, {b_n:.2f}; {b_s / b_n:.2f}; {capped}')

Q1: 578; 107, 35, 438, 181; 15; 578, 1
Q2: 0.0160; 561.5; 6251
Q3: 28.81, 48.17; 104.69, 35.00; 2.99; 10


## Q4-Q7(B): pure arithmetic (checked in code)

In [9]:
# ---- Q4: warmup + cosine, and the stepping-cadence bug ----
base = 3e-4
m = lambda t: t / 60 if t < 60 else 0.5 * (1 + math.cos(math.pi * (t - 60) / 420))
print('Q4:', f'{base*m(15):.2e}, {base*m(270):.2e}, {base*m(400):.2e}; {base*m(8):.2e}; scheduler.step() once per epoch = only 8 of 480 schedule steps -> LR never leaves the tiny warmup range (stuck near 4e-5), loss decreases very slowly / underfits')

# ---- Q5: early stopping with min_delta ----
sc = [.512, .601, .648, .661, .664, .668, .671, .670, .672, .669, .673, .665]
def early_stop(scores, patience=3, min_delta=0.0):
    best, best_e, bad = -1e9, 0, 0
    for e, s in enumerate(scores, 1):
        if s > best + min_delta: best, best_e, bad = s, e, 0
        else:
            bad += 1
            if bad >= patience: return e, best_e, best
    return None, best_e, best
e1 = early_stop(sc, 3, 0.002); e2 = early_stop(sc, 3, 0.0)
print('Q5:', f'{e1[0]}, {e1[1]}, {e1[2]:.3f}; {"yes" if e2[0] else "no"}, {e2[1]}')

# ---- Q6: memory and accumulation ----
P_ = 25_000_000; state = P_ * 4 * 4 / 1e6                       # weights + grads + 2 Adam moments, FP32
fp32 = lambda bs: state + 150 * bs; amp = lambda bs: state + 75 * bs
cmax = lambda per: int((3400 - state) // per)
micro, accum = 8, 4
print('Q6:', f'{state:.0f}; {fp32(16):.0f}, {amp(16):.0f}; {cmax(150)}, {cmax(75)}; {micro*accum}; {960//micro}, {960//micro//accum}; {accum}; {2*(960//micro//accum)}')
assert 960 % micro == 0 and (960 // micro) % accum == 0         # no partial batch / partial accumulation group

# ---- Q7 part B (toy) ----
fg, n_pos_, n_emp_, n_fa, blob = 0.50, 60, 340, 34, 1024
fp_ = 1 - (n_fa * blob / CANVAS) / n_emp_; ea_ = (n_emp_ - n_fa) / n_emp_
official = .7 * fg + .2 * fp_ + .1 * ea_
naive = (n_pos_ * fg + (n_emp_ - n_fa) * 1 + n_fa * 0) / (n_pos_ + n_emp_)
base_naive = n_emp_ / (n_pos_ + n_emp_)                          # all-empty baseline: positives 0, empties 1
print('Q7 B:', f'{official:.4f}, {naive:.4f}; {"Naive Dice" if naive < base_naive else "none"}, {base_naive:.4f}', '| official all-empty baseline = 0.3000 < model', f'{official:.4f}')

Q4: 7.50e-05, 1.50e-04, 2.61e-05; 4.00e-05; scheduler.step() once per epoch = only 8 of 480 schedule steps -> LR never leaves the tiny warmup range (stuck near 4e-5), loss decreases very slowly / underfits
Q5: 10, 7, 0.671; no, 11
Q6: 400; 2800, 1600; 20, 40; 32; 120, 30; 4; 60
Q7 B: 0.6398, 0.8400; Naive Dice, 0.8500 | official all-empty baseline = 0.3000 < model 0.6398


## Q7 part A and Q9: all-empty baseline and defect-size bins

In [10]:
# ---- Q7 part A ----
n_empty_pairs = int((~pos).sum())
print('Q7 A:', f'{n_empty_pairs}; {n_empty_pairs / pos.size:.4f}; {0.7*0 + 0.2*1 + 0.1*1:.4f}')

# ---- Q9: size bins over positive pairs (all classes pooled) ----
a_pos = area[pos]; bins = [(1, 100), (101, 500), (501, 10**9)]
n_bin = [int(((a_pos >= lo) & (a_pos <= hi)).sum()) for lo, hi in bins]
pix_share = a_pos[a_pos >= 501].sum() / a_pos.sum() * 100
recall = [0.30, 0.70, 0.95]; missed = [n * (1 - r) for n, r in zip(n_bin, recall)]
names = ['1-100 px', '101-500 px', '501+ px']
print('Q9:', f'{n_bin[0]}, {n_bin[1]}, {n_bin[2]}; {pix_share:.2f}; {sum(missed):.2f}, {names[int(np.argmax(missed))]}; many small defects each cost one missed pair but few pixels, while few large defects hold most of the pixels')

Q7 A: 4039; 0.8415; 0.3000
Q9: 5, 99, 657; 99.24; 66.05, 501+ px; many small defects each cost one missed pair but few pixels, while few large defects hold most of the pixels


## Q8 and Q10: four short training runs (R0, R0b, R1 = the Q8 run, R2)
Model = a U-Net from scratch (swap `make_model()` for **your Milestone 2/3 baseline** if it differs). Fixed: native 128x800, AdamW lr 3e-4, batch 16, AMP, seed 42, threshold 0.5, no min-area filter, **no augmentation** (match your baseline if it used any).
The cosine schedule of Q4 is rescaled to the run length: warmup = 1 epoch (60 steps), cosine over the remaining steps (300 for 6 epochs).

In [12]:
import torch, torch.nn as nn, torch.nn.functional as F
assert torch.cuda.is_available(), 'turn on a GPU'
DEV = 'cuda'; torch.backends.cudnn.benchmark = True
MEAN, STD = 0.449, 0.226; BS, EPOCHS = 16, 6
import cv2
from concurrent.futures import ThreadPoolExecutor
def load_img(i):
    img = cv2.imread(f'{DATA_DIR}/train/{i}.jpg', cv2.IMREAD_GRAYSCALE)
    if img is None:
        return np.zeros((128, 800), dtype=np.uint8) # fallback to a blank canvas of H, W if file is missing
    return img
with ThreadPoolExecutor(4) as ex: X = np.stack(list(ex.map(load_img, ids)))
Xg, Mg = torch.from_numpy(X).to(DEV), torch.from_numpy(M).to(DEV)          # whole subset on the GPU (about 0.6 GB)
tr_t, va_t = torch.from_numpy(tr_idx).to(DEV), torch.from_numpy(va_idx).to(DEV)

class ConvBlock(nn.Module):
    def __init__(s, i, o):
        super().__init__(); s.f = nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(True), nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(True))
    def forward(s, x): return s.f(x)
class UNet(nn.Module):
    def __init__(s, base=24, depth=4):
        super().__init__(); ch = [base * 2 ** i for i in range(depth + 1)]
        s.down = nn.ModuleList([ConvBlock(1, ch[0])] + [ConvBlock(ch[i], ch[i + 1]) for i in range(depth)])
        s.up = nn.ModuleList([nn.ConvTranspose2d(ch[i + 1], ch[i], 2, stride=2) for i in range(depth)])
        s.dec = nn.ModuleList([ConvBlock(2 * ch[i], ch[i]) for i in range(depth)]); s.head = nn.Conv2d(ch[0], 4, 1)
    def forward(s, x):
        sk = []
        for i, b in enumerate(s.down): x = b(x if i == 0 else F.max_pool2d(x, 2)); sk.append(x)
        for i in reversed(range(len(s.up))): x = s.dec[i](torch.cat([s.up[i](x), sk[i]], 1))
        return s.head(x)
make_model = lambda: UNet()

def loss_fn(lg, y):                                                          # BCE + soft Dice on pairs that contain the defect
    bce = F.binary_cross_entropy_with_logits(lg, y); p = torch.sigmoid(lg)
    inter, den = (p * y).sum((2, 3)), p.sum((2, 3)) + y.sum((2, 3)); ps = y.sum((2, 3)) > 0
    return bce + ((1 - (2 * inter + 1) / (den + 1))[ps].mean() if ps.any() else lg.sum() * 0)

def official_score(A, I, G):                                                 # arrays [n,4]: predicted area, intersection, truth area
    ps, em = G > 0, G == 0
    fg = (2 * I[ps] / (A[ps] + G[ps])).mean() if ps.any() else 0.0           # empty prediction on a positive pair -> I=0 -> Dice 0
    fp = 1 - (A[em] / CANVAS).mean(); ea = (A[em] == 0).mean()
    return 0.7 * fg + 0.2 * fp + 0.1 * ea

@torch.no_grad()
def evaluate(model, thr=0.5):
    model.eval(); A, I, G, tot, n = [], [], [], 0., 0
    for i in range(0, len(va_t), 48):
        ids_ = va_t[i:i + 48]; x = ((Xg[ids_].float() / 255 - MEAN) / STD).unsqueeze(1); y = Mg[ids_].float()
        with torch.autocast('cuda', dtype=torch.float16): lg = model(x)
        lg = lg.float(); tot += loss_fn(lg, y).item() * len(ids_); n += len(ids)
        pr = torch.sigmoid(lg) >= thr; gt = y > 0
        A.append(pr.sum((2, 3))); I.append((pr & gt).sum((2, 3))); G.append(gt.sum((2, 3)))
    A, I, G = (torch.cat(t).cpu().numpy().astype(np.float64) for t in (A, I, G))
    return tot / n, official_score(A, I, G), A, G

# bounded sqrt sampler weights computed on the 960 TRAINING images (Question 3 recipe)
cnt_tr = Counter(combo[tr_idx]); n_clean_tr = cnt_tr['clean']
w_img = torch.tensor([min(math.sqrt(n_clean_tr / cnt_tr[c]), 5.0) for c in combo[tr_idx]], dtype=torch.float, device=DEV)

def run_experiment(name, seed=42, cosine=False, sqrt_sampler=False):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    model = make_model().to(DEV); opt = torch.optim.AdamW(model.parameters(), lr=3e-4); scaler = torch.amp.GradScaler('cuda')
    spe = len(tr_t) // BS; total, warm = EPOCHS * spe, spe                                   # 60 steps per epoch
    sched = torch.optim.lr_scheduler.LambdaLR(opt, (lambda t: t / warm if t < warm else 0.5 * (1 + math.cos(math.pi * (t - warm) / (total - warm)))) if cosine else (lambda t: 1.0))
    if USE_WANDB:
        import wandb; wandb.init(project=WANDB_PROJECT, name=name, config=dict(seed=seed, cosine=cosine, sqrt_sampler=sqrt_sampler, lr=3e-4, bs=BS, epochs=EPOCHS, amp=True))
    best = dict(score=-1); hist = []
    for ep in range(1, EPOCHS + 1):
        model.train(); order = tr_t[torch.multinomial(w_img, len(tr_t), replacement=True)] if sqrt_sampler else tr_t[torch.randperm(len(tr_t), device=DEV)]
        run = 0.
        for k in range(spe):
            ids_ = order[k * BS:(k + 1) * BS]; x = ((Xg[ids_].float() / 255 - MEAN) / STD).unsqueeze(1); y = Mg[ids_].float()
            with torch.autocast('cuda', dtype=torch.float16): lg = model(x)
            loss = loss_fn(lg.float(), y); opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); run += loss.item()
        vl, vs, A, G = evaluate(model); hist.append(dict(epoch=ep, train_loss=run / spe, val_loss=vl, val_score=vs, lr=opt.param_groups[0]['lr']))
        print(f"{name} ep{ep} train {run/spe:.4f} val {vl:.4f} score {vs:.4f}")
        if USE_WANDB: wandb.log(hist[-1])
        if vs > best['score']: best = dict(score=vs, epoch=ep, state=copy.deepcopy(model.state_dict()))
    model.load_state_dict(best['state']); _, _, A, G = evaluate(model); em = G == 0
    best['fa_rate'] = float((A[em] > 0).mean() * 100)                                         # % empty pairs with >=1 predicted pixel
    if USE_WANDB: wandb.summary.update(dict(best_score=best['score'], best_epoch=best['epoch'], fa_rate=best['fa_rate'])); wandb.finish()
    return hist, best

In [13]:
t0 = time.time()
h0, b0 = run_experiment('R0_baseline_const_lr', seed=42)
h0b, b0b = run_experiment('R0b_seed7', seed=7)
h1, b1 = run_experiment('R1_warmup_cosine', seed=42, cosine=True)            # R1 is also the Q8 run
h2, b2 = run_experiment('R2_sqrt_sampler', seed=42, sqrt_sampler=True)
print(f'4 runs took {(time.time()-t0)/60:.1f} min')

# ---- Q8 (the R1 run: warmup + cosine) ----
hh = pd.DataFrame(h1); e_vl = int(hh.val_loss.idxmin()) + 1; e_vs = int(hh.val_score.idxmax()) + 1
gap = hh.val_loss.iloc[-1] - hh.train_loss.iloc[-1]
print('Q8:', f'{e_vl}; {e_vs}, {hh.val_score.max():.3f}; {gap:.3f}; monitor the official validation score (the metric you are judged on), keep epoch {e_vs}')
print(hh.round(4).to_string(index=False))
print('pattern hint -> val loss trend:', hh.val_loss.round(3).tolist(), '| train loss:', hh.train_loss.round(3).tolist(), '(falling together = still underfitting / healthy; val rising while train falls = overfitting)')

# ---- Q10 ----
noise = abs(b0['score'] - b0b['score']); g1, g2 = b1['score'] - b0['score'], b2['score'] - b0['score']
print('Q10:', f"{b0['score']:.3f}, {b0b['score']:.3f}, {b1['score']:.3f}, {b2['score']:.3f}; {noise:.3f}, {g1:.3f}, {g2:.3f}; {'yes' if g1 > 2*noise else 'no'}, {'yes' if g2 > 2*noise else 'no'}; {b0['fa_rate']:.1f}, {b2['fa_rate']:.1f}")
print('e) Changing several things at once confounds their effects, so the gain cannot be attributed to one component (OFAT isolates one factor per run).')
print('f) The sampler raises the share of defect images seen in training (prior shift), so the model fires more often and the false-alarm rate on clean images rises unless the threshold is re-calibrated.')

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 23f2004185 (23f2004185-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


R0_baseline_const_lr ep1 train 1.4924 val 0.0591 score 0.1551
R0_baseline_const_lr ep2 train 1.3978 val 0.0562 score 0.2626
R0_baseline_const_lr ep3 train 1.3327 val 0.0556 score 0.2697
R0_baseline_const_lr ep4 train 1.2712 val 0.0538 score 0.2710
R0_baseline_const_lr ep5 train 1.2184 val 0.0515 score 0.2712
R0_baseline_const_lr ep6 train 1.1791 val 0.0502 score 0.3000


... (5 outputs left out)

wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


R2_sqrt_sampler ep1 train 1.5081 val 0.0594 score 0.1619
R2_sqrt_sampler ep2 train 1.4126 val 0.0551 score 0.2130
R2_sqrt_sampler ep3 train 1.3423 val 0.0541 score 0.2684
R2_sqrt_sampler ep4 train 1.2824 val 0.0520 score 0.2711
R2_sqrt_sampler ep5 train 1.2326 val 0.0495 score 0.2712
R2_sqrt_sampler ep6 train 1.1913 val 0.0484 score 0.3000


epoch,▁▂▄▅▇█
lr,▁▁▁▁▁▁
train_loss,█▆▄▃▂▁
val_loss,█▅▅▃▂▁
val_score,▁▄▆▇▇█
best_epoch,6
best_score,0.3
epoch,6
fa_rate,0
lr,0.0003
train_loss,1.19129


4 runs took 9.1 min
Q8: 6; 6, 0.271; -1.243; monitor the official validation score (the metric you are judged on), keep epoch 6
 epoch  train_loss  val_loss  val_score     lr
     1      1.5501    0.0603     0.1492 0.0003
     2      1.4347    0.0560     0.1759 0.0003
     3      1.3683    0.0545     0.2506 0.0002
     4      1.3237    0.0528     0.2708 0.0001
     5      1.3001    0.0521     0.2710 0.0000
     6      1.2952    0.0518     0.2711 0.0000
pattern hint -> val loss trend: [0.06, 0.056, 0.054, 0.053, 0.052, 0.052] | train loss: [1.55, 1.435, 1.368, 1.324, 1.3, 1.295] (falling together = still underfitting / healthy; val rising while train falls = overfitting)
Q10: 0.300, 0.273, 0.271, 0.300; 0.027, -0.029, 0.000; no, no; 0.0, 0.0
e) Changing several things at once confounds their effects, so the gain cannot be attributed to one component (OFAT isolates one factor per run).
f) The sampler raises the share of defect images seen in training (prior shift), so the model fires mor